In [ ]:
from google.cloud import bigquery
import warnings
import pandas as pd
from datasets import Dataset, DatasetInfo
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

client = bigquery.Client(project="ads-599-final")

In [ ]:
# Outpatient measurement records from hosp.omr for cohort patients.
# Includes weight, height, and eGFR recorded outside of hospital admissions.
# Blood pressure excluded (captured in triage and ed.vitalsign).
# BMI excluded (derived from height and weight).
# Used to supplement ED and inpatient state features with recent baseline measurements.

query_omr = """
SELECT DISTINCT
  o.subject_id,
  o.chartdate,
  o.result_name,
  o.result_value
FROM `physionet-data.mimiciv_3_1_hosp.omr` AS o
WHERE o.subject_id IN (
  SELECT DISTINCT subject_id
  FROM `physionet-data.mimiciv_ed.edstays`
)
AND o.result_name NOT LIKE 'BMI%'
AND o.result_name NOT LIKE 'Blood Pressure%'
"""

print("Running OMR query...")
df_omr = client.query(query_omr).to_dataframe()
print(f"Shape: {df_omr.shape}")
print(f"Unique subjects: {df_omr['subject_id'].nunique():,}")
print(f"\nResult types:\n{df_omr['result_name'].value_counts()}")
df_omr.head()

In [ ]:
PHYSIONET_LICENSE = "PhysioNet Credentialed Health Data License 1.5.0 — https://physionet.org/content/mimiciv/view-license/3.1/"

info = DatasetInfo(
    description=(
        "Outpatient measurement records from hosp.omr for cohort patients — includes weight, "
        "height, and eGFR recorded outside of hospital admissions. "
        "Used to supplement ED and inpatient state features with recent baseline measurements."
    ),
    license=PHYSIONET_LICENSE,
)

ds = Dataset.from_pandas(df_omr, split='omr', info=info)
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="omr", data_dir="omr")
print("Pushed omr to HuggingFace Hub.")